# 29 · Review round 2: the pilot rule with patient-clustered inference, and maximum-rank versus mean-rank pairing (post hoc)

**Status: post hoc, exploratory.** Answers review round 2 (30 Oct 2026), points R2.1 and R2.2. Reads only frozen prediction files and writes aggregates only; BRSET stays in this runtime.

**Part A. The pilot rule, corrected (R2.1).** The rule in notebook 24 compared the two candidates' AUCs on the pilot with a DeLong test that treated every pilot image as independent, although several images can come from one patient (DDR: one filename group). Here:
- the pilot is drawn exactly as in notebook 24 (same random sequence, whole patients, every image of drawn patients removed from evaluation), so the always-pick, always-pair and DeLong rows reproduce notebook 24;
- **the corrected rule** compares the two AUCs with a **patient-clustered bootstrap** inside the pilot (200 resamples of pilot patients, target and other patients resampled separately): use the pilot winner alone if the 95% interval of the AUC difference excludes zero, otherwise pair. This is the rule stated in the manuscript, with its inference corrected; it is not re-chosen;
- **a comparison rule aligned with the endpoint**: each system's referral threshold is set at the operating share on the whole local population (unlabelled), the pilot counts how many target cases each system would miss at that threshold, and the same clustered bootstrap decides pick or pair. It is reported as a comparison, not substituted for the main rule;
- **mean-rank pairing** is added as a second way to pair, for always-pair and for both adaptive rules (R2.2);
- every policy is scored on the **same evaluation images** in each draw. The blind pick (either system with probability one half) is now computed on those images too, replacing the full-cohort baseline;
- uncertainty: intervals for each policy and for **paired differences between policies** (adaptive minus always pair, adaptive minus always pick, mean rank minus maximum rank) resample evaluation patients (DDR: filename groups), using the draw-averaged result for each image. Percentiles across pilot draws are reported separately and labelled as allocation variability;
- summaries: each roster; a **population-balanced mean** (rosters averaged within each of the five populations, then populations weighted equally); and the **largest observed roster mean** (not a worst case or a bound);
- pilots report the number of distinct patients (DDR: filename groups) as well as images.

**Part B. Maximum rank versus mean rank on the main rosters (R2.2).** For every pair in the regret rosters (all run combinations), at the operating share on the full test sets: missed cases per 10,000 images for the average member, the better member (chosen in each resample), the maximum-rank pair and the mean-rank pair, with patient-clustered intervals for the differences.

**Part C. Figures.** Redraws Figures 1, 2 and 7 and Supplementary Figure 11 for the round 2 edits, and adds Supplementary Figure 14 (maximum versus mean rank).

**Runtime:** CPU, High-RAM; about 60 to 90 minutes.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import os, re, json, hashlib, itertools, datetime, time, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
DATA = "/content/drive/MyDrive/stochastic_orthogonality/data"
BASE = f"{DATA}/dr_ensemble_v2"; PREDS, SPLITS = f"{BASE}/03_predictions", f"{BASE}/01_splits"
BRSET_LABELS = f"{DATA}/brset/label_brset.csv"; DERM = f"{DATA}/derm"
OUTD = f"{BASE}/04_results/revisions_c2"; os.makedirs(OUTD, exist_ok=True)
OUT = f"{BASE}/05_figures/manuscript_final"
OUT_JSON = f"{OUTD}/round2_policies_c2.json"
assert not os.path.exists(OUT_JSON), "already written; rename the old output deliberately to rerun"
B, SEED, DRAWS, KPILOT, BCL = 1000, 20261030, 100, (25, 50, 100), 200   # BCL: cluster-bootstrap resamples inside each pilot
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind="stable")[:int(round(f * len(s)))]] = True; return r
def rk(s): return rankdata(s) / len(s)
def pairscore(ra, rb): return np.maximum(ra, rb) + 1e-9 * np.minimum(ra, rb)
def boot_w(groups, n_boot, seed):
    codes = pd.factorize(groups)[0]; G = codes.max() + 1; rng = np.random.default_rng(seed)
    W = np.empty((n_boot + 1, len(codes)), np.float32); W[0] = 1.0
    for b in range(1, n_boot + 1): W[b] = np.bincount(rng.integers(0, G, G), minlength=G)[codes]
    return W
def ci(x):
    x = np.asarray(x, float); f = x[1:][np.isfinite(x[1:])]
    return [float(x[0]), float(np.percentile(f, 2.5)), float(np.percentile(f, 97.5))]
def ci_basic(x):
    x = np.asarray(x, float); f = x[1:][np.isfinite(x[1:])]
    return [float(x[0]), float(2 * x[0] - np.percentile(f, 97.5)), float(2 * x[0] - np.percentile(f, 2.5))]
def pct(x): x = np.asarray(x, float); return [float(np.mean(x)), float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))]
def fmt(t, d=2): return f"{t[0]:.{d}f} ({t[1]:.{d}f} to {t[2]:.{d}f})"
def pick(cols, pats, what):
    m = [c for c in cols if any(re.fullmatch(p, c, flags=re.I) for p in pats)]
    assert len(m) == 1, f"cannot identify the {what} column; columns are {list(cols)}"
    return m[0]
def norm(x): return pd.Series(x, dtype=str).str.replace(r"\.(jpe?g|png|tiff?)$", "", regex=True, flags=re.I).to_numpy()
HASHES = {}
print("ready")
from scipy.stats import norm as _norm


Mounted at /content/drive
ready


In [2]:
ARCHS = ["densenet121_binary", "densenet121_5class", "efficientnet_b3_binary", "efficientnet_b3_5class", "resnet50_binary", "resnet50_5class",
         "vit_base_binary", "vit_base_5class", "retfound_binary", "retfound_5class", "retfound_adversarial"]
CORE4 = ["densenet121_5class", "efficientnet_b3_5class", "resnet50_5class", "retfound_5class"]
RSEEDS = [0, 1, 2]; RMODELS = [(a, k) for a in ARCHS for k in RSEEDS]
def tag(k): return "" if k == 0 else f"_seed{k}"
def load_preds(fmt_):
    P = {}; ref = None
    for a, k in RMODELS:
        fn = f"{PREDS}/{fmt_.format(arch=a, tag=tag(k))}"; HASHES[os.path.basename(fn)] = sha(fn)
        d = pd.read_csv(fn, dtype={"image_id": str})
        if ref is None: ref = d
        else: assert (d.image_id.values == ref.image_id.values).all(), fn
        P[(a, k)] = d["predicted_probability"].to_numpy(np.float64)
    return ref, P
RET = {}
ref, P = load_preds("eyepacs_predictions_fresh_{arch}{tag}_v2.csv")
RET["EyePACS"] = {"grade": ref.true_grade.to_numpy(int), "group": ref.patient_id.astype(str).to_numpy(), "P": P, "f": 0.25}
ref, P = load_preds("ddr_predictions_{arch}{tag}_v2.csv")
sp = pd.read_csv(f"{SPLITS}/ddr_split_intraset_v2.csv", dtype={"image_id": str})
gcol, grpcol = pick(sp.columns, ["grade", "true_grade", "dr_grade", "level"], "DDR grade"), pick(sp.columns, ["group", "filename_group", "group_id"], "DDR group")
sp["_k"] = norm(sp["image_id"]); sp = sp.set_index("_k").loc[norm(ref.image_id)]
g = sp[gcol].to_numpy(int); keep = g <= 4; isA = sp["half"].astype(str).str.upper().eq("A").to_numpy() if "half" in sp.columns else np.ones(len(g), bool)
RET["DDR"] = {"grade": g[keep], "group": sp[grpcol].astype(str).to_numpy()[keep], "P": {m: v[keep] for m, v in P.items()}, "f": round(1.31 * float((g[keep & isA] >= 2).mean()), 4)}
ref, P = load_preds("brset_predictions_eyepacs_{arch}{tag}_v2.csv")
lb = pd.read_csv(BRSET_LABELS, dtype=str)
icol, pcol, idcol = pick(lb.columns, [r".*icdr.*"], "BRSET ICDR grade"), pick(lb.columns, ["patient_id", "patient"], "BRSET patient"), pick(lb.columns, ["image_id", "image"], "BRSET image id")
lb["_k"] = norm(lb[idcol]); lb = lb.set_index("_k").loc[norm(ref.image_id)]
g = pd.to_numeric(lb[icol], errors="coerce").to_numpy(); keep = np.isin(g, [0, 1, 2, 3, 4])
RET["BRSET"] = {"grade": g[keep].astype(int), "group": lb[pcol].astype(str).to_numpy()[keep], "P": {m: v[keep] for m, v in P.items()}}
RET["BRSET"]["f"] = round(1.31 * float((RET["BRSET"]["grade"] >= 2).mean()), 4)
PB = {}
for src in ("eyepacs", "ddr"):
    for a in CORE4:
        for k in RSEEDS:
            fn = f"{PREDS}/brset_predictions_{src}_{a}{tag(k)}_v2.csv"; HASHES[os.path.basename(fn)] = sha(fn)
            d = pd.read_csv(fn, dtype={"image_id": str}); assert (norm(d.image_id) == lb.index.to_numpy()).all(), fn
            PB[(src, a, k)] = d["predicted_probability"].to_numpy(np.float64)[keep]
for n_, d in RET.items(): print(f"{n_:>8}: {len(d['grade']):,} images | {len(set(d['group'])):,} groups | f = {d['f']}")
# skin
CLASSES = ["MEL", "BCC", "SCC", "AK", "NV", "BKL", "DF", "VASC"]; SARCHS = ["resnet50", "densenet121", "efficientnet_b3", "convnext_t", "vit_b", "vit_b_dinov2"]
SOURCES = ["HAM10000", "BCN20000"]; SMODELS = [(s, a, k) for s in SOURCES for a in SARCHS for k in (0, 1)]; VARIANTS = [(s, a) for s in SOURCES for a in SARCHS]
SKIN = {}
for es in ("HIBA", "PAD"):
    D = pd.read_csv(f"{DERM}/manifest_{es}.csv", dtype={"image_id": str, "patient_id": str}).sort_values("image_id")
    D = D[~D.duplicated("raw_sha256", keep="first") & D.dx.isin(CLASSES)].reset_index(drop=True); P = {}
    for m in SMODELS:
        fn = f"{DERM}/predictions/{es}__{m[0]}_{m[1]}_seed{m[2]}.csv"; HASHES[os.path.basename(fn)] = sha(fn)
        p = pd.read_csv(fn, dtype={"image_id": str}).drop_duplicates("image_id").set_index("image_id").reindex(D.image_id); assert p.p_malignant.notna().all(); P[m] = p.p_malignant.to_numpy()
    dx = D.dx.to_numpy(); mal = np.isin(dx, ["MEL", "BCC", "SCC"])
    SKIN[es] = {"dx": dx, "mal": mal, "mel": dx == "MEL", "group": D.patient_id.astype(str).fillna("none").to_numpy(), "P": P, "f": round(1.31 * float(mal.mean()), 4)}
    print(f"{es:>8}: {len(dx):,} images | {len(set(SKIN[es]['group'])):,} patients | f = {SKIN[es]['f']}")

 EyePACS: 15,794 images | 7,897 groups | f = 0.25
     DDR: 12,422 images | 332 groups | f = 0.555
   BRSET: 16,266 images | 8,524 groups | f = 0.0745
    HIBA: 1,616 images | 623 patients | f = 0.6088
     PAD: 2,283 images | 1,373 patients | f = 0.6203


In [3]:
def mname(m): return "_".join(str(x) for x in m)
ROSTERS = []
main_pairs = [((a, i), (b, j), "main roster") for a, b in itertools.combinations(ARCHS, 2) for i in RSEEDS for j in RSEEDS]
for pop in ("EyePACS", "DDR", "BRSET"):
    d = RET[pop]
    ROSTERS.append({"key": f"{pop}_severe", "label": f"{pop}, severe DR", "y": d["grade"] >= 3, "group": d["group"], "P": d["P"], "pairs": main_pairs, "f": d["f"]})
ROSTERS.append({"key": "DDR_pdr", "label": "DDR, PDR", "y": RET["DDR"]["grade"] == 4, "group": RET["DDR"]["group"], "P": RET["DDR"]["P"], "pairs": main_pairs, "f": RET["DDR"]["f"]})
def src_type(a, b):
    if a[0] == b[0]: return f"both {'EyePACS' if a[0] == 'eyepacs' else 'DDR'}-trained"
    return "one of each"
mixed_pairs = [(a, b, src_type(a, b)) for a, b in itertools.combinations(PB, 2) if a[:2] != b[:2]]
ROSTERS.append({"key": "BRSET_mixed_severe", "label": "BRSET, mixed training sources, severe DR", "y": RET["BRSET"]["grade"] >= 3, "group": RET["BRSET"]["group"], "P": PB, "pairs": mixed_pairs, "f": RET["BRSET"]["f"]})
skin_pairs = [((*va, i), (*vb, j), "same collection" if va[0] == vb[0] else "different collection") for va, vb in itertools.combinations(VARIANTS, 2) for i in (0, 1) for j in (0, 1)]
for es in ("HIBA", "PAD"):
    S = SKIN[es]
    for oname, y in (("melanoma", S["mel"]), ("any skin cancer", S["mal"])):
        ROSTERS.append({"key": f"{es}_{'mel' if oname == 'melanoma' else 'mal'}", "label": f"{es}, {oname}", "y": y, "group": S["group"], "P": S["P"], "pairs": skin_pairs, "f": S["f"]})
for R_ in ROSTERS: print(f"{R_['key']:22s} pairs {len(R_['pairs']):4d} | cases {int(R_['y'].sum()):5d} | f {R_['f']}")

EyePACS_severe         pairs  495 | cases   712 | f 0.25
DDR_severe             pairs  495 | cases  1149 | f 0.555
BRSET_severe           pairs  495 | cases   474 | f 0.0745
DDR_pdr                pairs  495 | cases   913 | f 0.555
BRSET_mixed_severe     pairs  252 | cases   474 | f 0.0745
HIBA_mel               pairs  264 | cases   253 | f 0.6088
HIBA_mal               pairs  264 | cases   751 | f 0.6088
PAD_mel                pairs  264 | cases    52 | f 0.6203
PAD_mal                pairs  264 | cases  1081 | f 0.6203


## A1. Pilot draws, clustered tests and policies

In [4]:
def refer_fast(score, f):
    r = int(round(f * len(score))); out = np.zeros(len(score), bool)
    if r > 0: out[np.argpartition(-score, r - 1)[:r]] = True
    return out
def meanscore(ra, rb): return (ra + rb) / 2 + 1e-9 * np.maximum(ra, rb)
def delong_parts(scores, ycal):
    pos, neg = ycal == 1, ycal == 0; V10, V01, A = {}, {}, {}
    for m, s in scores.items():
        X, Yn = s[pos][:, None], s[neg][None, :]; psi = (X > Yn) + 0.5 * (X == Yn)
        V10[m], V01[m] = psi.mean(1), psi.mean(0); A[m] = float(V10[m].mean())
    return A, V10, V01
def delong_p(a, b, V10, V01):
    d10, d01 = V10[a] - V10[b], V01[a] - V01[b]; var = d10.var(ddof=1) / len(d10) + d01.var(ddof=1) / len(d01)
    if var <= 0: return 1.0 if d10.mean() == 0 else 0.0
    return float(2 * _norm.sf(abs(d10.mean()) / np.sqrt(var)))
def cluster_w(codes, nb, rng_):
    G = codes.max() + 1; W = np.empty((nb + 1, len(codes)), np.float32); W[0] = 1.0
    for b in range(1, nb + 1): W[b] = np.bincount(rng_.integers(0, G, G), minlength=G)[codes]
    return W
def clear(d):   # d: (BCL+1,) difference; True if the 95% percentile interval excludes zero
    q = d[1:]; return bool((q > 0).mean() >= 0.975 or (q < 0).mean() >= 0.975)
POL = ["better", "blind", "pick_auc", "pick_vol", "pair_max", "pair_mean", "adapt_delong_max", "adapt_auc_max", "adapt_auc_mean", "adapt_vol_max", "adapt_vol_mean"]
ACC, DRAWSTAT, PILOT = {}, {}, {}
for R_ in ROSTERS:
    t0 = time.time(); y = R_["y"]; pid = pd.factorize(R_["group"])[0]; npat = pid.max() + 1; P = R_["P"]; f = R_["f"]; N = len(y)
    used = sorted({m for a, b, _ in R_["pairs"] for m in (a, b)}, key=str); npairs = len(R_["pairs"])
    THR = {m: np.sort(P[m])[::-1][max(int(round(f * N)) - 1, 0)] for m in used}          # operating threshold on the whole unlabelled population
    tpat = np.unique(pid[y]); opat = np.setdiff1d(np.arange(npat), tpat); rng = np.random.default_rng(SEED + 1); rngb = np.random.default_rng(SEED + 7)
    ACC[R_["key"]], DRAWSTAT[R_["key"]], PILOT[R_["key"]] = {}, {}, {}
    for k in KPILOT:
        if y.sum() < 2 * k: continue
        acc = {p: np.zeros(N, np.float64) for p in POL}; cnt = np.zeros(N, np.float64)
        ds = {p: np.zeros(DRAWS) for p in POL}; share = {p: np.zeros(DRAWS) for p in POL if p.startswith("adapt")}; pil = []
        for dr in range(DRAWS):
            cs, ni = [], 0
            for p_ in rng.permutation(tpat):
                cs.append(p_); ni += int((y & (pid == p_)).sum())
                if ni >= k: break
            cn, nj = [], 0
            for p_ in rng.permutation(opat):
                cn.append(p_); nj += int((pid == p_).sum())
                if nj >= k: break
            si = rng.permutation(np.flatnonzero(y & np.isin(pid, cs)))[:k]; ni_ = rng.permutation(np.flatnonzero(np.isin(pid, cn)))[:k]
            cal = np.r_[si, ni_]; ycal = np.r_[np.ones(len(si)), np.zeros(len(ni_))]
            pil.append((len(np.unique(pid[si])), len(np.unique(pid[ni_])), len(si), len(ni_)))
            A, V10, V01 = delong_parts({m: P[m][cal] for m in used}, ycal)
            Wp = cluster_w(pd.factorize(pid[si])[0], BCL, rngb); Wn = cluster_w(pd.factorize(pid[ni_])[0], BCL, rngb)
            AB, MB = {}, {}
            for m in used:
                sp, sn = P[m][si], P[m][ni_]; psi = (sp[:, None] > sn[None, :]).astype(np.float32) + 0.5 * (sp[:, None] == sn[None, :])
                AB[m] = ((Wp @ psi) * Wn).sum(1) / (Wp.sum(1) * Wn.sum(1)); MB[m] = Wp @ (sp < THR[m]).astype(np.float32)
            ev = ~np.isin(pid, np.r_[cs, cn]); idx = np.flatnonzero(ev); ye = y[ev]; ne = ev.sum(); cnt[idx] += 1
            MV = {m: ye & ~refer(P[m][ev], f) for m in used}; K = {m: rk(P[m][ev]) for m in used}; MC = {m: int(MV[m].sum()) for m in used}
            tot = {p: np.zeros(ne, np.float64) for p in POL}; nad = {p: 0 for p in share}
            for a, b, _ in R_["pairs"]:
                va, vb = MV[a], MV[b]; vx = ye & ~refer_fast(pairscore(K[a], K[b]), f); vm = ye & ~refer_fast(meanscore(K[a], K[b]), f)
                lo = va if MC[a] <= MC[b] else vb
                da = AB[a] - AB[b]; dv = MB[a] - MB[b]
                win_auc = va if da[0] > 0 else (vb if da[0] < 0 else None)
                win_vol = va if dv[0] < 0 else (vb if dv[0] > 0 else win_auc)
                blind = 0.5 * (va.astype(np.float64) + vb)
                pa = win_auc if win_auc is not None else blind; pv = win_vol if win_vol is not None else blind
                c_auc, c_vol, c_dl = clear(da), clear(dv), delong_p(a, b, V10, V01) < 0.05
                ch = {"better": lo, "blind": blind, "pick_auc": pa, "pick_vol": pv, "pair_max": vx, "pair_mean": vm,
                      "adapt_delong_max": pa if c_dl else vx, "adapt_auc_max": pa if c_auc else vx, "adapt_auc_mean": pa if c_auc else vm,
                      "adapt_vol_max": pv if c_vol else vx, "adapt_vol_mean": pv if c_vol else vm}
                for p in POL: tot[p] += ch[p]
                nad["adapt_delong_max"] += c_dl; nad["adapt_auc_max"] += c_auc; nad["adapt_auc_mean"] += c_auc; nad["adapt_vol_max"] += c_vol; nad["adapt_vol_mean"] += c_vol
            for p in POL:
                acc[p][idx] += tot[p] / npairs; ds[p][dr] = (tot[p].sum() - tot["better"].sum()) / npairs / ne * 1e4
            for p in share: share[p][dr] = 1 - nad[p] / npairs
        ACC[R_["key"]][k] = {"acc": acc, "cnt": cnt}; DRAWSTAT[R_["key"]][k] = {"draws": ds, "share_paired": share}
        pil = np.array(pil); PILOT[R_["key"]][k] = {"target_patients_mean": float(pil[:, 0].mean()), "target_patients_min": int(pil[:, 0].min()), "other_patients_mean": float(pil[:, 1].mean()),
                                                    "target_images_mean": float(pil[:, 2].mean()), "other_images_mean": float(pil[:, 3].mean())}
        print(f"{R_['key']:22s} k={k:3d} | pilot: {PILOT[R_['key']][k]['target_patients_mean']:.1f} target patients (min {PILOT[R_['key']][k]['target_patients_min']}) for {PILOT[R_['key']][k]['target_images_mean']:.0f} target images ({time.time() - t0:.0f}s)")

EyePACS_severe         k= 25 | pilot: 15.2 target patients (min 13) for 25 target images (77s)
EyePACS_severe         k= 50 | pilot: 30.1 target patients (min 26) for 50 target images (154s)
EyePACS_severe         k=100 | pilot: 60.0 target patients (min 55) for 100 target images (237s)
DDR_severe             k= 25 | pilot: 4.3 target patients (min 1) for 25 target images (64s)
DDR_severe             k= 50 | pilot: 5.4 target patients (min 1) for 50 target images (129s)
DDR_severe             k=100 | pilot: 11.8 target patients (min 3) for 100 target images (201s)
BRSET_severe           k= 25 | pilot: 16.1 target patients (min 13) for 25 target images (77s)
BRSET_severe           k= 50 | pilot: 31.3 target patients (min 26) for 50 target images (155s)
BRSET_severe           k=100 | pilot: 63.5 target patients (min 58) for 100 target images (238s)
DDR_pdr                k= 25 | pilot: 1.7 target patients (min 1) for 25 target images (66s)
DDR_pdr                k= 50 | pilot: 3.1 target

## A2. Results: each roster, the population-balanced mean and the largest observed roster mean
Regret is missed cases per 10,000 evaluation images above the better member of each pair (chosen with hindsight on the same images). Intervals resample evaluation patients (DDR: filename groups) using each image's draw-averaged contribution; "draws" gives the 2.5th to 97.5th percentiles across pilot draws (allocation variability given these data).

In [5]:
DIFFS = [("adapt_auc_max", "pair_max"), ("adapt_auc_max", "pick_auc"), ("adapt_auc_max", "blind"), ("adapt_vol_max", "pair_max"), ("adapt_vol_max", "adapt_auc_max"),
         ("pair_mean", "pair_max"), ("adapt_auc_mean", "adapt_auc_max"), ("adapt_delong_max", "adapt_auc_max")]
RESA = {}
for R_ in ROSTERS:
    key = R_["key"]; RESA[key] = {"label": R_["label"]}
    if not ACC[key]: continue
    W = boot_w(R_["group"], B, SEED + 3)
    for k, a_ in ACC[key].items():
        den = W @ a_["cnt"]; reg = {p: (W @ (a_["acc"][p] - a_["acc"]["better"])) / den * 1e4 for p in POL}
        o = {p: ci(reg[p]) for p in POL if p != "better"}
        o["draws"] = {p: pct(DRAWSTAT[key][k]["draws"][p]) for p in POL if p != "better"}
        o["share_paired"] = {p: float(v.mean()) for p, v in DRAWSTAT[key][k]["share_paired"].items()}
        o["diff"] = {f"{p}_minus_{q}": ci(reg[p] - reg[q]) for p, q in DIFFS}
        o["pilot"] = PILOT[key][k]; RESA[key][str(k)] = o
        print(f"{key:22s} k={k:3d} | blind {fmt(o['blind'])} | pick {fmt(o['pick_auc'])} | pair max {fmt(o['pair_max'])} | pair mean {fmt(o['pair_mean'])} | adaptive (clustered AUC) {fmt(o['adapt_auc_max'])}, paired {o['share_paired']['adapt_auc_max']:.0%} | adaptive (DeLong, notebook 24) {o['adapt_delong_max'][0]:.2f}")
        print(f"{'':22s}        adaptive minus pair {fmt(o['diff']['adapt_auc_max_minus_pair_max'])} | adaptive minus pick {fmt(o['diff']['adapt_auc_max_minus_pick_auc'])} | endpoint-aligned rule {fmt(o['adapt_vol_max'])} | mean minus max (pair) {fmt(o['diff']['pair_mean_minus_pair_max'])}")
POPOF = lambda key: key.split("_")[0]
SUMA = {}
for k in map(str, KPILOT):
    keys = [kk for kk in RESA if k in RESA[kk]]; pops = sorted({POPOF(kk) for kk in keys})
    S_ = {}
    for p in [q for q in POL if q != "better"]:
        vals = {kk: RESA[kk][k][p][0] for kk in keys}
        popm = {pp: float(np.mean([v for kk, v in vals.items() if POPOF(kk) == pp])) for pp in pops}
        mx = max(vals, key=vals.get)
        S_[p] = {"roster_mean": float(np.mean(list(vals.values()))), "population_balanced_mean": float(np.mean(list(popm.values()))), "by_population": popm,
                 "largest_observed_roster_mean": [float(vals[mx]), mx]}
    SUMA[k] = {"rosters": keys, "populations": pops, "policies": S_}
    print(f"\nk={k} ({len(keys)} rosters, {len(pops)} populations): population-balanced mean regret (largest observed roster mean)")
    for p in ("blind", "pick_auc", "pair_max", "pair_mean", "adapt_delong_max", "adapt_auc_max", "adapt_auc_mean", "adapt_vol_max", "adapt_vol_mean"):
        v = S_[p]; print(f"   {p:18s} {v['population_balanced_mean']:7.2f} ({v['largest_observed_roster_mean'][0]:.2f}, {v['largest_observed_roster_mean'][1]}) | roster mean {v['roster_mean']:.2f}")

EyePACS_severe         k= 25 | blind 3.63 (2.46 to 4.84) | pick 2.93 (2.01 to 3.94) | pair max -2.91 (-4.18 to -1.76) | pair mean -2.45 (-3.69 to -1.08) | adaptive (clustered AUC) -2.61 (-3.83 to -1.52), paired 94% | adaptive (DeLong, notebook 24) -2.79
                              adaptive minus pair 0.30 (0.23 to 0.37) | adaptive minus pick -5.54 (-6.76 to -4.33) | endpoint-aligned rule -2.81 (-4.06 to -1.69) | mean minus max (pair) 0.46 (-0.85 to 1.76)
EyePACS_severe         k= 50 | blind 3.47 (2.37 to 4.64) | pick 2.31 (1.61 to 3.15) | pair max -2.84 (-4.02 to -1.72) | pair mean -2.34 (-3.50 to -1.06) | adaptive (clustered AUC) -2.49 (-3.55 to -1.48), paired 90% | adaptive (DeLong, notebook 24) -2.61
                              adaptive minus pair 0.35 (0.23 to 0.48) | adaptive minus pick -4.80 (-5.86 to -3.71) | endpoint-aligned rule -2.65 (-3.76 to -1.60) | mean minus max (pair) 0.50 (-0.73 to 1.76)
EyePACS_severe         k=100 | blind 3.22 (2.21 to 4.33) | pick 1.61 (1.12 to 

## B. Maximum-rank versus mean-rank pairs on the main rosters (full test sets)

In [6]:
RESB = {}
for R_ in ROSTERS:
    t0 = time.time(); y = R_["y"]; P = R_["P"]; f = R_["f"]; N = len(y); W = boot_w(R_["group"], B, SEED + 5); Wt = W.sum(1)
    used = sorted({m for a, b, _ in R_["pairs"] for m in (a, b)}, key=str); col = {m: i for i, m in enumerate(used)}
    MVm = np.column_stack([(y & ~refer(P[m], f)) for m in used]).astype(np.float32); MS = W @ MVm          # (B+1, models)
    K = {m: rk(P[m]) for m in used}
    PX = np.column_stack([(y & ~refer(pairscore(K[a], K[b]), f)) for a, b, _ in R_["pairs"]]).astype(np.float32)
    PMn = np.column_stack([(y & ~refer(meanscore(K[a], K[b]), f)) for a, b, _ in R_["pairs"]]).astype(np.float32)
    SX, SM = W @ PX, W @ PMn
    ia = np.array([col[a] for a, b, _ in R_["pairs"]]); ib = np.array([col[b] for a, b, _ in R_["pairs"]])
    A_, B_ = MS[:, ia], MS[:, ib]; better = np.minimum(A_, B_).mean(1); avg = ((A_ + B_) / 2).mean(1)
    mx, mn = SX.mean(1), SM.mean(1); per = lambda v: v / Wt * 1e4
    o = {"average_member": ci(per(avg)), "better_member": ci(per(better)), "max_rank_pair": ci(per(mx)), "mean_rank_pair": ci(per(mn)),
         "max_rank_avoided_vs_better": ci(per(better - mx)), "mean_rank_avoided_vs_better": ci(per(better - mn)),
         "max_rank_avoided_vs_average": ci(per(avg - mx)), "mean_rank_avoided_vs_average": ci(per(avg - mn)),
         "mean_minus_max_avoided": ci(per(mx - mn)), "share_pairs_mean_rank_misses_fewer": float((SM[0] < SX[0]).mean()),
         "share_pairs_mean_rank_misses_more": float((SM[0] > SX[0]).mean()), "pairs": len(R_["pairs"])}
    RESB[R_["key"]] = {"label": R_["label"], **o}
    print(f"{R_['key']:22s} vs better: max {fmt(o['max_rank_avoided_vs_better'])} | mean {fmt(o['mean_rank_avoided_vs_better'])} | mean minus max {fmt(o['mean_minus_max_avoided'])} | mean fewer in {o['share_pairs_mean_rank_misses_fewer']:.0%} of pairs ({time.time() - t0:.0f}s)")

EyePACS_severe         vs better: max 3.00 (1.45 to 3.92) | mean 2.52 (0.90 to 3.56) | mean minus max -0.48 (-1.81 to 0.84) | mean fewer in 34% of pairs (4s)
DDR_severe             vs better: max 3.79 (1.49 to 5.71) | mean 1.71 (-0.41 to 3.26) | mean minus max -2.07 (-3.83 to -0.70) | mean fewer in 22% of pairs (3s)
BRSET_severe           vs better: max -1.83 (-3.86 to -0.32) | mean -2.06 (-4.79 to 0.59) | mean minus max -0.22 (-2.35 to 2.28) | mean fewer in 60% of pairs (3s)
DDR_pdr                vs better: max 3.04 (0.87 to 5.00) | mean 0.91 (-1.14 to 2.33) | mean minus max -2.13 (-3.90 to -0.77) | mean fewer in 21% of pairs (2s)
BRSET_mixed_severe     vs better: max -8.17 (-10.86 to -5.69) | mean -9.19 (-13.37 to -5.52) | mean minus max -1.03 (-4.69 to 2.41) | mean fewer in 50% of pairs (2s)
HIBA_mel               vs better: max -6.56 (-23.97 to 0.23) | mean 4.52 (-12.35 to 10.86) | mean minus max 11.09 (2.69 to 19.84) | mean fewer in 69% of pairs (0s)
HIBA_mal               vs bet

## Write the results record

In [7]:
REC = {"status": "post hoc, exploratory (review round 2, R2.1 and R2.2); no lock", "written_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
       "pilot_rule": "use the pilot AUC winner alone if the 95% patient-clustered bootstrap interval (200 resamples) of the AUC difference excludes zero, otherwise pair; endpoint-aligned comparison uses pilot misses at the population operating threshold",
       "draws": DRAWS, "evaluation_bootstrap": B, "A_policies": RESA, "A_summary": SUMA, "B_max_vs_mean_rank": RESB, "input_sha256": HASHES}
def tolist(x): return {str(k): tolist(v) for k, v in x.items()} if isinstance(x, dict) else ([tolist(v) for v in x] if isinstance(x, (list, tuple)) else (x.tolist() if isinstance(x, np.ndarray) else x))
with open(OUT_JSON, "w") as fh: json.dump(tolist(REC), fh, indent=1, default=float)
print("written", OUT_JSON)

written /content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/04_results/revisions_c2/round2_policies_c2.json


## C. Figures 1, 2 and 7, Supplementary Figures 11 and 14

In [8]:
# Shared style
import shutil
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#333333', 'axes.linewidth': 0.8, 'xtick.color': '#333333', 'ytick.color': '#333333',
                     'axes.labelcolor': '#222222', 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.labelsize': 8.5, 'legend.frameon': False})
BLUE, SKY, ORANGE, VERM, PINK, GREY, INK, GREEN = '#0072B2', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7', '#8C8C8C', '#222222', '#009E73'
DOMCOL = {'retina': BLUE, 'dermatology': VERM}
SITES = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']
SNAME = {'EyePACS': 'EyePACS (US)', 'DDR': 'DDR (China)', 'BRSET': 'BRSET (Brazil)', 'HIBA': 'HIBA (Argentina)', 'PAD': 'PAD-UFES-20 (Brazil)'}
SDOM = {'EyePACS': 'retina', 'DDR': 'retina', 'BRSET': 'retina', 'HIBA': 'dermatology', 'PAD': 'dermatology'}
MARK = {'EyePACS': 'o', 'DDR': 's', 'BRSET': 'P', 'HIBA': '^', 'PAD': 'D'}
LS = {'EyePACS': '-', 'DDR': '--', 'BRSET': ':', 'HIBA': '-', 'PAD': '--'}
CNAME = {'moderate': 'Moderate NPDR', 'severe NPDR': 'Severe NPDR', 'PDR': 'PDR', 'BCC': 'BCC', 'SCC': 'SCC', 'melanoma': 'Melanoma'}
# relationship of two systems: one display name and one colour, used in Figures 3, 5 and 6
RELMAP = {'different design, different data': 'Different design, different data', 'different design, same data': 'Different design, same data',
          'different design': 'Different design, same data', 'same checkpoint, different data': 'Same design, different data',
          'variant sibling': 'Variant sibling (same design and data)', 'seed sibling': 'Seed sibling (repeat run)'}
RELS = ['Different design, different data', 'Different design, same data', 'Same design, different data', 'Variant sibling (same design and data)', 'Seed sibling (repeat run)']
RELCOL = dict(zip(RELS, [BLUE, SKY, ORANGE, PINK, VERM]))
def ci_txt(m, lo, hi, d=2, pct=False):
    u = '%' if pct else ''
    return f'{m:.{d}f}{u} ({lo:.{d}f} to {hi:.{d}f})'
def panel(ax, letter, text, sub=None):
    ax.set_title(f'{letter}  {text}', loc='left', fontsize=9.5, weight='bold', pad=14 if sub else 6)
    if sub: ax.text(0, 1.015, sub, transform=ax.transAxes, fontsize=8, color='#444444', va='bottom')
def title(fig, t): fig.suptitle(t, x=0.01, ha='left', fontsize=11, weight='bold')
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M')
def save(fig, name):
    for ext in ('png', 'pdf'):
        p = f'{OUT}/{name}.{ext}'
        if os.path.exists(p):
            dest = f'{OUT}/SUPERSEDED_{name}.{ext}'
            if os.path.exists(dest): dest = f'{OUT}/SUPERSEDED_{name}_{STAMP}.{ext}'
            shutil.move(p, dest)
    fig.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)
DR = json.load(open(f'{DERM}/derm_results_v1.json'))
RESULTS = f'{BASE}/04_results'
REV = f'{RESULTS}/revisions_c1'
REG = json.load(open(f'{REV}/regret_c1.json'))['results']; ADA = json.load(open(f'{REV}/adaptive_rule_c1.json'))['results']
DEP = json.load(open(f'{REV}/deployment_c1.json')); POOL = json.load(open(f'{REV}/pooled_vs_pairing_c1.json'))
PAIRS = pd.read_csv(f'{REV}/regret_c1_pairs.csv'); BRR = json.load(open(f'{RESULTS}/brset_results_v6.json'))
from matplotlib.patches import FancyBboxPatch
def box(ax, x, y, w, h, head, body, ec, fc, hs=10, bs=8.3, lw=1.4, ital=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.004,rounding_size=0.01', ec=ec, fc=fc, lw=lw, transform=ax.transAxes))
    if head: ax.text(x + w / 2, y + h - 0.016, head, ha='center', va='top', fontsize=hs, weight='bold', color=INK, transform=ax.transAxes)
    if body: ax.text(x + w / 2, (y + h - 0.05) if head else (y + h / 2), body, ha='center', va='top' if head else 'center', fontsize=bs, color=INK,
                     linespacing=1.35, style='italic' if ital else 'normal', transform=ax.transAxes)
def arrow(ax, x0, y0, x1, y1, col='#666666'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), xycoords='axes fraction', arrowprops=dict(arrowstyle='-|>', color=col, lw=1.3, shrinkA=0, shrinkB=0))
def gtag(ax, x, y, t='confirmatory, dated plan'):
    ax.text(x, y, t, ha='center', va='center', fontsize=7.4, color='white', weight='bold', transform=ax.transAxes, bbox=dict(boxstyle='round,pad=0.3', fc=GREEN, ec='none'))
def blank(ax): ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
LBL = {'EyePACS_severe': 'EyePACS, severe DR', 'DDR_severe': 'DDR, severe DR', 'DDR_pdr': 'DDR, PDR', 'BRSET_severe': 'BRSET, severe DR',
       'BRSET_mixed_severe': 'BRSET, mixed sources, severe DR', 'HIBA_mel': 'HIBA, melanoma', 'HIBA_mal': 'HIBA, any skin cancer',
       'PAD_mel': 'PAD-UFES-20, melanoma', 'PAD_mal': 'PAD-UFES-20, any skin cancer'}
LBL_DEP = {**LBL, 'DDR_pdr': 'DDR, PDR'}
KC = {'EyePACS_severe': '#0072B2', 'DDR_severe': '#56B4E9', 'DDR_pdr': '#009E73', 'BRSET_severe': '#003A70', 'BRSET_mixed_severe': '#8C8C8C',
      'HIBA_mel': '#D55E00', 'HIBA_mal': '#E69F00', 'PAD_mel': '#CC79A7', 'PAD_mal': '#8B4513'}
KM = lambda k: '^' if k.split('_')[0] in ('HIBA', 'PAD') else 'o'
def p0(x): return f'{int(np.floor(100 * x + 0.5))}%'   # percentages rounded half up, as in the text

FD = json.load(open(f'{OUT}/figures_c1_data.json')); DET = FD['sibling_detector']; SITES5 = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']


In [9]:
# Figure 1: study design
fig = plt.figure(figsize=(17, 9.2)); ax = fig.add_axes([0, 0, 1, 0.95]); blank(ax)
RF, RE, SF, SE = '#EAF2FA', BLUE, '#FCF1E4', VERM
ax.text(0.30, 0.955, 'Diabetic retinopathy', ha='center', fontsize=15, weight='bold', color=BLUE, transform=ax.transAxes)
ax.text(0.81, 0.955, 'Skin cancer', ha='center', fontsize=15, weight='bold', color=VERM, transform=ax.transAxes)
ax.plot([0.607, 0.607], [0.21, 0.93], color='#C8C8C8', lw=1.2, transform=ax.transAxes)
# retina training and models
box(ax, 0.03, 0.78, 0.26, 0.12, 'Training: EyePACS (US screening)', '13,532 images; 2,900 for\ncheckpoint selection; split by patient', RE, RF)
box(ax, 0.33, 0.78, 0.25, 0.12, 'Training: DDR (China)', 'official training file, 6,210 images;\n2,487 for checkpoint selection', RE, RF)
box(ax, 0.03, 0.56, 0.26, 0.15, '33 models', '11 designs x 3 training runs\nDenseNet, EfficientNet, ResNet, ViT\n(2- and 5-class); 3 RETFound variants', RE, RF)
box(ax, 0.33, 0.56, 0.25, 0.15, '12 models', '4 five-class designs x 3 runs\n(DenseNet, EfficientNet, ResNet,\nRETFound); identical recipe', RE, RF)
arrow(ax, 0.16, 0.78, 0.16, 0.71); arrow(ax, 0.455, 0.78, 0.455, 0.71)
TX = [0.01, 0.155, 0.30, 0.445]; TW = 0.135
TESTS = [('EyePACS', 'test set\n15,794 eyes\n7,897 patients\nnever used\nin training'), ('DDR', '12,422 images\nChinese\nhospital\nexternal'),
         ('EyePACS', 'partner-choice\npools\n35,718 and\n17,858 images'), ('BRSET', 'Brazil\n16,266 images\n8,524 patients\nall 45 models;\nexternal to all')]
for x, (h, b) in zip(TX, TESTS):
    box(ax, x, 0.26, TW, 0.2, h, b, RE, RF, hs=10.5, bs=8.4); gtag(ax, x + TW / 2, 0.232, 'confirmatory, dated plan' if h != 'EyePACS' or 'test set' in b else 'confirmatory (partner choice), dated plan')
for x in TX: arrow(ax, 0.16, 0.56, x + TW / 2, 0.46)
arrow(ax, 0.455, 0.56, TX[3] + TW / 2 + 0.01, 0.46)
# skin
box(ax, 0.63, 0.78, 0.17, 0.12, 'HAM10000', 'dermoscopy\nAustria, Australia\n11,720 images', SE, SF)
box(ax, 0.82, 0.78, 0.17, 0.12, 'BCN20000', 'dermoscopy\nSpain\n18,946 images', SE, SF)
box(ax, 0.665, 0.56, 0.29, 0.15, '24 models', '6 designs x 2 training collections x 2 runs\nResNet, DenseNet, EfficientNet, ConvNeXt,\nViT (supervised), ViT (DINOv2)', SE, SF)
arrow(ax, 0.715, 0.78, 0.76, 0.71); arrow(ax, 0.905, 0.78, 0.86, 0.71)
box(ax, 0.63, 0.26, 0.17, 0.2, 'HIBA', 'Argentina\n1,616 images\n623 patients\ndermoscopic and clinical', SE, SF, hs=10.5, bs=8.4)
box(ax, 0.82, 0.26, 0.17, 0.2, 'PAD-UFES-20', 'Brazil, smartphone\n2,283 images\n1,373 patients\ncancers biopsy-proven', SE, SF, hs=10.5, bs=8.4)
arrow(ax, 0.76, 0.56, 0.715, 0.46); arrow(ax, 0.86, 0.56, 0.905, 0.46)
gtag(ax, 0.715, 0.232); gtag(ax, 0.905, 0.232)
box(ax, 0.03, 0.03, 0.94, 0.15, 'Rules applied to every comparison',
    'A pair always refers the same number of images as one system. Confirmatory analyses followed a plan dated before their data were examined; later, exploratory analyses reused the same predictions.\n'
    'Referral share: 25% in EyePACS; elsewhere 1.31 x the proportion of referable cases (DDR 56%, BRSET 7.5%, HIBA 61%, PAD 62%). Intervals resample patients (image groups in DDR).\n'
    'Comparators: the average member (either system, strength unknown) and the better member (chosen with hindsight).', '#9A9A9A', '#F4F4F4', hs=11, bs=9.2)
title(fig, 'Figure 1. Study design'); save(fig, 'Fig1_design')

saved Fig1_design


In [10]:
# Figure 2: benefit at the same referral volume; (a) against the average member, (b) against the better member on a count scale, (c) skin strongest partner
H1 = DR['H1_benefit']; b2 = BRR['B2_benefit']['a_eyepacs_all_110_pairings']['sev']
def pct_(r, k='relative_reduction_matched_volume'): return 100 * r[k], 100 * r['ci95'][0], 100 * r['ci95'][1]
ROWSA = [('EyePACS (US)\nPDR (secondary)', (40, 30, 51), 'retina', False), ('DDR (China)\nProliferative DR', (27, 19, 40), 'retina', True),
         ('BRSET (Brazil)\nSevere DR (grade 3 or 4)', pct_(b2, 'relative_reduction'), 'retina', True), ('HIBA (Argentina)\nAny skin cancer', pct_(H1['HIBA_f']['mal']), 'dermatology', True),
         ('HIBA (Argentina)\nMelanoma', pct_(H1['HIBA_f']['mel']), 'dermatology', True), ('PAD-UFES-20 (Brazil)\nAny skin cancer', pct_(H1['PAD_f']['mal']), 'dermatology', True)]
NEY, NDDR = 15794, 12422
ROWSB = [('EyePACS (US)\nSevere DR, pre-specified primary', tuple(v / NEY * 1e4 for v in (4.74, 2.31, 6.15)), 'retina', True),
         ('DDR (China)\nProliferative DR', tuple(v / NDDR * 1e4 for v in (3.796, 1.547, 6.39)), 'retina', False),
         ('BRSET (Brazil)\nSevere DR', (-1.8, -3.9, -0.3), 'retina', False),
         ('HIBA (Argentina)\nAny skin cancer', (26.4, 2.5, 41.6), 'dermatology', False), ('HIBA (Argentina)\nMelanoma', (-7.0, -24.4, 6.4), 'dermatology', False),
         ('PAD-UFES-20 (Brazil)\nAny skin cancer', (21.8, -7.1, 40.2), 'dermatology', False), ('PAD-UFES-20 (Brazil)\nMelanoma', (-6.5, -17.9, -1.8), 'dermatology', False)]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6), gridspec_kw={'width_ratios': [1, 1.05, 0.9]})
def forest(ax, rows, unit, xl, fmtv):
    for i, (lab, (m, lo, hi), dom, pre) in enumerate(rows):
        c = DOMCOL[dom]; ax.plot([lo, hi], [i, i], color=c, lw=2.4, solid_capstyle='round'); ax.plot(m, i, 'o', ms=8.5, color=c if pre else 'white', mec=c, mew=1.8, zorder=3)
        ax.text(max(hi, m, 0) + xl * 0.02, i, fmtv(m, lo, hi), va='center', fontsize=8.3)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8); ax.set_xlabel(unit)
forest(axes[0], ROWSA, 'Reduction in missed cases vs the average member (%)', 60, lambda m, lo, hi: f'{m:.0f}% ({lo:.0f} to {hi:.0f})'); axes[0].set_xlim(-2, 62)
panel(axes[0], 'a', 'Against the average member (relative)', 'Filled: pre-specified; open: secondary')
forest(axes[1], ROWSB, 'Missed cases avoided vs the better member, per 10,000 images', 60, lambda m, lo, hi: f'{m:.1f} ({lo:.1f} to {hi:.1f})'); axes[1].set_xlim(-28, 52)
panel(axes[1], 'b', 'Against the better member, chosen with hindsight (count scale)', 'Filled: pre-specified primary (4.74 net per pair); open: secondary.\nBelow 0, the pair misses more')
ax = axes[2]
SP = [('HIBA (Argentina)\nAny skin cancer', (17, 12, 22), (27, 18, 37)), ('HIBA (Argentina)\nMelanoma', (7, 2, 12), (26, 14, 40)), ('PAD-UFES-20 (Brazil)\nAny skin cancer', (10, 7, 12), (20, 14, 26))]
for i, (lab, avg, best) in enumerate(SP):
    for (m, lo, hi), dy, mk, c in ((avg, -0.17, 'o', GREY), (best, 0.17, 'D', VERM)):
        ax.plot([lo, hi], [i + dy] * 2, color=c, lw=2.4, solid_capstyle='round'); ax.plot(m, i + dy, mk, ms=8, color=c, mec='white', mew=0.8, zorder=3)
        if mk == 'D': ax.text(hi + 1.2, i + dy, f'{m:.0f}% ({lo:.0f} to {hi:.0f})', va='center', fontsize=8.3, color=VERM)
ax.set_yticks(range(len(SP))); ax.set_yticklabels([s[0] for s in SP]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8); ax.set_xlim(-2, 55)
ax.set_xlabel('Reduction in missed cases vs the average member (%)')
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=GREY, ms=7, label='average over all pairs'), Line2D([], [], marker='D', ls='', color=VERM, ms=7, label='strongest partner (own validation score)')],
          fontsize=7.6, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=2)
panel(ax, 'c', 'Exploratory: the strongest partner (skin)')
title(fig, 'Figure 2. A second reader at the same referral volume, against the average member and against the better member (95% intervals resample patients or image groups)')
fig.tight_layout(); save(fig, 'Fig2_benefit')

saved Fig2_benefit


In [11]:
# Figure 7: an exploratory decision framework, with regret panels from Part A (matched evaluation cohorts)
K7 = '100'; SA = SUMA[K7]['policies']
STR7 = [('blind', 'Pick one blind (same images)', GREY, 'o'), ('pick_auc', 'Always use the pilot winner', ORANGE, 's'), ('pair_max', 'Always pair', BLUE, 'D'), ('adapt_auc_max', 'Pilot test, then pick or pair', GREEN, '^')]
pb = lambda p: SA[p]['population_balanced_mean']; lw_ = lambda p: SA[p]['largest_observed_roster_mean'][0]
rank_fl = [DET[s]['rank_corr']['siblings_flagged'] for s in SITES5]; sib_bm = BRR['B4_sibling_test']['repeat_runs_flagged']
fig = plt.figure(figsize=(17.5, 10.4))
ax = fig.add_axes([0.0, 0.03, 0.57, 0.9]); blank(ax)
ax.text(0.81, 0.985, 'Evidence in this study', ha='center', fontsize=10.5, weight='bold', color='#666666', transform=ax.transAxes)
STEPS = [('1  Judge a second system at the referral volume you can absorb',
          'A pair refers the same number of images as one system. Referring\nevery case either system flags catches more, but only by referring more.',
          'Same volume: fewer misses than the average member\nin all five populations; against the better member,\nthe result depended on population and outcome.', BLUE, '#EAF2FA'),
         ('2  Weigh partner strength as well as diversity',
          'Different training data means fewer shared misses, but a weaker\npartner can make a strong system worse.',
          'Fewer shared misses with different data (pre-specified,\nboth domains). Added catch at equal strength is unresolved;\nin BRSET, a weaker DDR-trained partner left EyePACS-\ntrained systems 10.6 more misses per 10,000.', BLUE, '#EAF2FA'),
         ('3  Test the candidates on local cases, then pick or pair',
          'Draw about 100 severe cases and 100 others from many patients.\nIf one system is clearly stronger (patient-clustered interval\nfor the AUC difference), use it alone; otherwise run both.',
          f'Mean regret per 10,000 at 100 + 100, population-balanced\n(largest roster mean): pilot test {pb("adapt_auc_max"):.1f} ({lw_("adapt_auc_max"):.1f});\nalways pair {pb("pair_max"):.1f} ({lw_("pair_max"):.1f}); always pick {pb("pick_auc"):.1f} ({lw_("pick_auc"):.1f}).\nDeveloped on these data; needs independent validation.', GREEN, '#E7F4EF'),
         ('4  Check for close relatives on unlabelled local images',
          "Compare how the candidates rank the programme's own images;\nflag agreement above that among systems known to differ.",
          f'Pre-specified kappa rule flagged {p0(sib_bm)} of BRSET siblings\n(target 80%); score ranks flagged {p0(min(rank_fl))} to {p0(max(rank_fl))}\nacross populations (exploratory, revised after the test).', BLUE, '#EAF2FA'),
         ('5  Choose the referral share for local capacity and prevalence',
          'The share used here kept referrals in proportion to local\nreferable disease; other shares were not validated.',
          'Neither pre-specified test found the predicted reversal\n(DDR at a copied 25% share; EyePACS enriched to 40.6%);\nbenefit held across shares (Supplementary Figs 1 and 2).', BLUE, '#EAF2FA')]
y0, hh, gap = 0.80, 0.135, 0.045
for i, (h, b, ev, ec, fc) in enumerate(STEPS):
    y = y0 - i * (hh + gap)
    box(ax, 0.0, y, 0.6, hh, h, b, ec, fc, hs=10, bs=8.6, lw=1.6 if ec == GREEN else 1.4)
    box(ax, 0.63, y + 0.008, 0.365, hh - 0.016, None, ev, '#BDBDBD', '#F7F7F7', bs=7.7, lw=1.0, ital=True)
    ax.plot([0.6, 0.63], [y + hh / 2] * 2, color='#BDBDBD', lw=1, transform=ax.transAxes)
    if i < len(STEPS) - 1: arrow(ax, 0.3, y, 0.3, y - gap + 0.002, col=BLUE)
ax.text(0.0, y - 0.035, 'Expect the most benefit for conditions each system rarely misses; where all systems miss the same cases\n(melanoma among moles), plan another kind of second look.',
        fontsize=8.4, color='#444444', va='top', transform=ax.transAxes)
ax.text(0.0, 1.0, 'a  An exploratory framework for choosing and checking a second AI reader (not a validated protocol)', fontsize=10.5, weight='bold', va='bottom', transform=ax.transAxes)
gs = fig.add_gridspec(3, 2, left=0.68, right=0.99, top=0.9, bottom=0.11, hspace=0.75, wspace=0.32)
PTC = {'main roster': (BLUE, 'o', 'pair'), 'both EyePACS-trained': (BLUE, 'o', 'both EyePACS-trained'), 'one of each': (GREEN, '^', 'one of each'), 'both DDR-trained': (VERM, 's', 'both DDR-trained')}
for j, (key, lt, ttl) in enumerate((('EyePACS_severe', 'b', 'Close candidates: EyePACS'), ('BRSET_mixed_severe', 'c', 'Far apart: BRSET, mixed sources'))):
    a_ = fig.add_subplot(gs[0, j]); d = PAIRS[PAIRS.roster == key]; mx = d.gap_per_10k.max() * 1.05
    a_.fill_between([0, mx], [0, mx / 2], [0, mx], color='#DDDDDD', alpha=0.5, lw=0); a_.plot([0, mx], [0, mx / 2], ls='--', color='#777777', lw=1); a_.axhline(0, color='#333333', lw=0.8)
    for t, g_ in d.groupby('pair_type'):
        c, mk, lab = PTC[t]; a_.scatter(g_.gap_per_10k, g_.pair_regret_per_10k, s=9, alpha=0.5, color=c, marker=mk, lw=0, label=lab, rasterized=True)
    a_.set_xlabel('Gap between the two systems\n(missed per 10,000)', fontsize=7.8); a_.set_ylabel('Pair regret (missed per 10,000)' if j == 0 else '', fontsize=7.8)
    a_.set_title(f'{lt}  {ttl}', loc='left', fontsize=9, weight='bold')
    if j == 0: a_.legend(handles=[Line2D([], [], ls='--', color='#777777', label='blind pick, expected'), Patch(color='#DDDDDD', label='up to the worst-case blind pick'),
                                  Line2D([], [], ls='', marker='o', color=BLUE, ms=4, label='pair (below 0: beats the better system)')], fontsize=6.4, loc='upper left', handletextpad=0.3)
    if j == 1: a_.legend(fontsize=6.6, loc='upper left', markerscale=1.6, handletextpad=0.2)
for row, (lt, keys, ttl) in enumerate((('d', ['EyePACS_severe', 'DDR_severe', 'DDR_pdr', 'BRSET_severe', 'BRSET_mixed_severe'], 'Retina: regret by strategy, pilot 100 + 100, same evaluation images'),
                                      ('e', ['HIBA_mel', 'HIBA_mal', 'PAD_mal'], 'Skin: regret by strategy, pilot 100 + 100, same evaluation images')), start=1):
    a_ = fig.add_subplot(gs[row, :]); a_.axvline(0, color='#333333', lw=0.8); keys = [k for k in keys if K7 in RESA.get(k, {})]
    for i, k in enumerate(keys):
        for (p, lab, c, mk), dy in zip(STR7, (-0.27, -0.09, 0.09, 0.27)):
            m, lo, hi = RESA[k][K7][p]; a_.plot([lo, hi], [i + dy] * 2, color=c, lw=1.2); a_.scatter(m, i + dy, color=c, marker=mk, s=30, edgecolor='white', lw=0.5, zorder=3, label=lab if i == 0 else None)
    a_.set_yticks(range(len(keys))); a_.set_yticklabels([LBL[k].replace(', ', '\n', 1) for k in keys], fontsize=7.6); a_.invert_yaxis(); a_.tick_params(axis='y', length=0)
    a_.set_xlabel('Regret: missed per 10,000 above the better system (0 = knowing which is stronger); 95% intervals resample patients', fontsize=7.6)
    a_.set_title(f'{lt}  {ttl}', loc='left', fontsize=9, weight='bold')
    if row == 2: a_.legend(fontsize=7.2, loc='upper center', bbox_to_anchor=(0.45, -0.34), ncol=4, handletextpad=0.2, columnspacing=1.0)
title(fig, 'Figure 7. Choosing and checking a second AI reader: an exploratory framework')
save(fig, 'Fig7_decision_flow')

saved Fig7_decision_flow


In [12]:
# Supplementary Figure 11: the pilot rule with patient-clustered inference (Part A)
KS = [k for k in ('25', '50', '100') if k in SUMA]
SN = [('pick_auc', 'Always use the pilot winner', ORANGE, 's', '-'), ('pair_max', 'Always pair (maximum rank)', BLUE, 'D', '-'), ('pair_mean', 'Always pair (mean rank)', SKY, 'D', '--'),
      ('adapt_auc_max', 'Pilot test (clustered AUC), then pick or pair', GREEN, '^', '-'), ('adapt_vol_max', 'Endpoint-aligned pilot test, then pick or pair', PINK, 'v', '-'), ('blind', 'Pick one blind', GREY, 'o', ':')]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.9), gridspec_kw={'width_ratios': [1, 1, 1.2]})
for a_, fld, lt, ttl in ((axes[0], 'population_balanced_mean', 'a', 'Population-balanced mean regret'), (axes[1], 'largest_observed_roster_mean', 'b', 'Largest observed roster mean')):
    for s, lab, c, mk, ls in SN:
        v = [SUMA[k]['policies'][s][fld] for k in KS]; v = [x[0] if isinstance(x, list) else x for x in v]
        a_.plot([int(k) for k in KS], v, marker=mk, color=c, ls=ls, lw=1.8, ms=6, mec='white', label=lab)
    a_.axhline(0, color='#333333', lw=0.8); a_.set_xticks([int(k) for k in KS]); a_.set_xticklabels([f'{k} + {k}' for k in KS])
    a_.set_xlabel('Pilot size (severe cases + others)'); a_.set_ylabel('Regret (missed per 10,000)'); panel(a_, lt, ttl)
axes[0].legend(fontsize=7.4, loc='upper center', bbox_to_anchor=(1.1, -0.17), ncol=3)
a_ = axes[2]; keys = [kk for kk in RESA if '100' in RESA[kk]]
for i, kk in enumerate(keys):
    for (d, c, dy, lab) in (('adapt_auc_max_minus_pair_max', GREEN, -0.15, 'pilot test minus always pair'), ('adapt_auc_max_minus_pick_auc', ORANGE, 0.15, 'pilot test minus always pick')):
        m, lo, hi = RESA[kk]['100']['diff'][d]; a_.plot([lo, hi], [i + dy] * 2, color=c, lw=1.6); a_.plot(m, i + dy, 'o', color=c, ms=5, label=lab if i == 0 else None)
a_.axvline(0, color='#333333', lw=0.8); a_.set_yticks(range(len(keys))); a_.set_yticklabels([LBL[k] for k in keys], fontsize=7.6); a_.invert_yaxis()
a_.set_xlabel('Difference in regret per 10,000 (below 0: the pilot test does better)'); a_.legend(fontsize=7, loc='lower right'); panel(a_, 'c', 'Paired differences at 100 + 100, same evaluation images')
title(fig, 'Supplementary Figure 11. A local pilot, then pick or pair, with patient-clustered inference (exploratory; scored on patients outside the pilot; 100 draws)')
fig.tight_layout(); save(fig, 'SuppFig11_pilot_then_pick_or_pair')

saved SuppFig11_pilot_then_pick_or_pair


In [13]:
# Supplementary Figure 14: maximum-rank versus mean-rank pairs (Part B)
keys = list(RESB)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={'width_ratios': [1.25, 1]})
a_ = axes[0]
for i, kk in enumerate(keys):
    for f_, c, dy, lab in (('max_rank_avoided_vs_better', BLUE, -0.15, 'maximum rank'), ('mean_rank_avoided_vs_better', SKY, 0.15, 'mean rank')):
        m, lo, hi = RESB[kk][f_]; a_.plot([lo, hi], [i + dy] * 2, color=c, lw=1.8); a_.plot(m, i + dy, 'D', color=c, ms=5.5, mec='white', label=lab if i == 0 else None)
a_.axvline(0, color='#333333', lw=0.8); a_.set_yticks(range(len(keys))); a_.set_yticklabels([LBL[k] for k in keys], fontsize=7.8); a_.invert_yaxis()
a_.set_xlabel('Missed cases avoided vs the better member, per 10,000 images'); a_.legend(fontsize=7.4, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2); panel(a_, 'a', 'Against the better member (chosen with hindsight)')
a_ = axes[1]
for i, kk in enumerate(keys):
    m, lo, hi = RESB[kk]['mean_minus_max_avoided']; a_.plot([lo, hi], [i, i], color=INK, lw=1.8); a_.plot(m, i, 'o', color=INK, ms=5.5)
    a_.text(max(hi, 0) + 0.5, i, f"{100 * RESB[kk]['share_pairs_mean_rank_misses_fewer']:.0f}% of pairs", va='center', fontsize=7.2, color='#555555')
a_.axvline(0, color='#333333', lw=0.8); a_.set_yticks(range(len(keys))); a_.set_yticklabels([''] * len(keys)); a_.invert_yaxis()
a_.set_xlabel('Mean rank minus maximum rank: extra misses avoided per 10,000'); panel(a_, 'b', 'Mean rank against maximum rank', 'Text: share of pairs in which mean rank missed fewer')
title(fig, 'Supplementary Figure 14. The benefit depends on how the two scores are combined (all pairs and run combinations; 95% intervals resample patients)')
fig.tight_layout(); save(fig, 'SuppFig14_max_vs_mean_rank')
drive.flush_and_unmount(); drive.mount('/content/drive'); print('done')

saved SuppFig14_max_vs_mean_rank
Mounted at /content/drive
done
